## LSTM Pipeline — Black-Box Modelling of Multiband Saturation

Here we perform the whole LSTM pipeline for the multiband saturation dataset, which includes:

- Imports
- Configuration 
- Dataset checking
- Split Train
- LSTM Model
- Loss Functions
- Optimizer + Scheduler
- Training Loop
- Test Set Evaluation (against baselines and plucking styles)
- JSON Export
- Audio Rendering

The outputs of this notebook are saved in the SAVE_DIR directory, which is defined in the configuration cell.

## Cell 1 — Imports

In [10]:
import os
import json
import time
import shutil
import random
import numpy as np
import soundfile as sf
from pathlib import Path
from IPython.display import Audio
from collections import defaultdict

import torch
import torchaudio
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import auraloss.time as auraloss_time
import auraloss.freq as auraloss_freq

print("All libraries imported!")

All libraries imported!


In [11]:
# Check if CUDA is available
device = torch.device('cpu')
if torch.cuda.is_available():
    device = torch.device('cuda')

torch.set_default_device(device)
DEVICE = str(device)
print(f"Using device = {torch.get_default_device()}")

print(f"Pytorch version: {torch.__version__}")
print(f"Torchaudio version: {torchaudio.__version__}")

Using device = cpu
Pytorch version: 2.11.0+cpu
Torchaudio version: 2.11.0+cpu


In [12]:
def set_deterministic_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_deterministic_seeds(42)
print("Deterministic seeds set to 42")

Deterministic seeds set to 42


## Cell 2 - Configuration

To activate the training and test cells, set `TRAIN = True` and `TEST = True`.

In [13]:
TRAIN = False
TEST = True

In [ ]:
# PATHS
CLEAN_DIR = r"Path\to\your\diretory\Dataset\clean"
SAT_DIR   = r"Path\to\your\diretory\Dataset\saturated"
RESULTS_DIR   = 'Results/Results_H96'   # (for hidden size 96)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Audio
SAMPLE_RATE   = 44100
CHUNK_SAMPLES = 8192                         # ~186ms a 44.1kHz

# train/val/test split
TRAIN_RATIO   = 0.7
VAL_RATIO     = 0.15
TEST_RATIO    = 1 - TRAIN_RATIO - VAL_RATIO

# LSTM Model
HIDDEN_SIZE   = 96                           # try 32, 64, 96
SKIP_CON      = 1                            
NUM_LAYERS    = 2                            

# Training
EPOCHS        = 800                           
BATCH_SIZE    = 16                           # try 16, 40
LEARNING_RATE = 5e-4 
SEGMENT_LEN   = 22050                        # 22050 equals to ~0.5 seconds at 44.1kHz
INIT_LEN      = 0
UP_FR         = 2048                         # MUST be >= 2048 for auraloss (MRSTFT) to work properly (since it uses a 2048 FFT window)
VAL_CHUNK     = 100000                       
VALIDATION_F  = 2                            # validates at N epochs
PATIENCE      = 25                           # early stopping
WEIGHT_DECAY  = 1e-5                         # L2 regularization para o Adam optimizer
THRESHOLD     = 1e-4                         # threshold to ignore silent chunks in ESR Loss

# Pre-emphasis filter (high-pass)
PRE_FILT = [-0.85, 1]

# Weight losses for the combined loss function
LAMBDA_ESR    = 0.45
LAMBDA_MRSTFT = 0.45
LAMBDA_DC     = 0.10

# If running in Colab, set WORKERS to 2 or 4 for faster data loading. If running on Windows + Jupyter, set to 0 to avoid issues with multiprocessing.
WORKERS = 0

## Cell 3 — Check Dataset and clean/saturated pair files

In [15]:
clean_files      = sorted(Path(CLEAN_DIR).rglob("*.wav"))
saturated_files  = sorted(Path(SAT_DIR).rglob("*.wav"))

print(f"Files found: {len(clean_files) + len(saturated_files)}")

sat_index = {
    f.stem.removesuffix("_saturated"): f
    for f in saturated_files
}

pair_files = []
missing    = []
ignored  = []

for file in clean_files:
    y_c, sr_clean = torchaudio.load(str(file))
    duration = y_c.shape[-1] / sr_clean
    if duration > 6:
        ignored.append(file.name)
        continue
    sat_file = sat_index.get(file.stem)
    if sat_file is None:
        missing.append(file.name)
        continue
    pair_files.append((file, sat_file))

print(f"Valid pairs         : {len(pair_files)}")

if missing:
    print(f"Missing pairs       : {len(missing)}")
    for m in missing:
        print(f"  ✗ {m}")

print(f"Ignored files (>6s): {len(ignored)}")

if not missing and len(pair_files) == len(clean_files) == len(saturated_files) or len(pair_files) != 0:
    print("\nDataset valid and ready!")
else:
    print("\nThere are samples without the corresponding pair.")

Files found: 3746
Valid pairs         : 1872
Ignored files (>6s): 1

Dataset valid and ready!


## Cell 4 — Split train

Since the dataset includes recordings from three distinct instruments, a random split could lead to optimistic results due to timbral leakage across sets. The model might learn the instrument characteristics instead of the intended saturation mapping.

To mitigate this, we adopted a 1-1-1 Split where each instrument is exclusively assigned to one subset (train, validation, or test). This enforces complete timbral separation, providing a strict evaluation of generalization.

In [16]:
def total_minutes(pairs):
    total_seconds = 0.0
    for clean_path, _ in pairs:
        y_c, sr_clean = torchaudio.load(str(clean_path))
        total_seconds += y_c.shape[-1] / sr_clean

    hours = int(total_seconds // 3600)
    mins = int((total_seconds % 3600) // 60)
    secs = total_seconds % 60
    if hours > 0:
        return f"{hours}h {mins}m {round(secs)}s", round(total_seconds)
    elif mins > 0:
        return f"{mins}m {round(secs)}s", round(total_seconds)
    return f"{secs:.1f}s", round(total_seconds)


def set_split_ratio(train, validation, test, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    """
    Defines the ratio of samples to keep for each set (validation/test)
    assuming the provided 'train' set represents 'train_ratio' (default 70%) of a total dataset.
    """
    random.seed(42)

    train_shuffled = random.sample(train, len(train))
    validation_shuffled = random.sample(validation, len(validation))
    test_shuffled = random.sample(test, len(test))

    train_final = train_shuffled

    val_target_len = round(len(train_shuffled) * (val_ratio / train_ratio))
    test_target_len = round(len(train_shuffled) * (test_ratio / train_ratio))

    validation_final = validation_shuffled[:val_target_len]
    test_final = test_shuffled[:test_target_len]

    return train_final, validation_final, test_final


def get_style_stats(pairs_list):
    """Auxiliary function to compute expression style stats for a list of pairs."""
    style_pairs = defaultdict(list)
    for c_path, s_path in pairs_list:
        stem = Path(c_path).stem
        parts = stem.split('_')
        # Assuming BS_3_EQ_2_FS_NO_1_0.wav, index 4 is the expression style
        if len(parts) > 4 and parts[4] in ['FS', 'MU', 'PK', 'ST']:
            style = parts[4]
        else:
            style = 'UNKNOWN'
        style_pairs[style].append((c_path, s_path))

    stats = {}
    for style, s_pairs in style_pairs.items():
        dur_str, sec = total_minutes(s_pairs)
        stats[style] = {'count': len(s_pairs), 'duration': dur_str, 'duration_sec': sec}
    return stats


def split_by_instrument(pair_files, train_guitar='1', val_guitar='2', test_guitar='3', set_plit=False):
    """
    Split clean/saturated pairs by guitar ID to prevent timbre overlap.
    Filename expected: BS_{k}_EQ_{m}_{plucking}_{expression}_{string}_{fret}.wav
    """
    buckets = defaultdict(list)

    for clean_path, sat_path in pair_files:
        stem = Path(clean_path).stem
        parts = stem.split('_')
        guitar_id = parts[1]
        buckets[guitar_id].append((clean_path, sat_path))

    train_pairs = []
    val_pairs   = []
    test_pairs  = []

    for guitar_id, pairs in buckets.items():
        if guitar_id == train_guitar:
            train_pairs.extend(pairs)
        elif guitar_id == val_guitar:
            val_pairs.extend(pairs)
        else:
            test_pairs.extend(pairs)

    if set_plit:
        train_pairs, val_pairs, test_pairs = set_split_ratio(train_pairs, val_pairs, test_pairs, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15)

    if not train_pairs or not val_pairs or not test_pairs:
        raise ValueError("One of the sets is empty. Please check the guitar IDs provided.")

    # time duration of each set
    train_duration, train_sec = total_minutes(train_pairs)
    val_duration, val_sec = total_minutes(val_pairs)
    test_duration, test_sec = total_minutes(test_pairs)
    total_duration_sec = train_sec + val_sec + test_sec

    # Stats by expression style
    train_stats = get_style_stats(train_pairs)
    val_stats = get_style_stats(val_pairs)
    test_stats = get_style_stats(test_pairs)

    print(f"Dataset duration: {total_duration_sec//60}min {total_duration_sec%60:02d}s\n")

    def print_split_info(split_name, guitar, pairs_len, duration, stats):
        print(f"{split_name} (guitar {guitar}): {pairs_len} pairs  ({duration})")
        for style, s_data in stats.items():
            print(f"  - {style}: {s_data['count']} files, {s_data['duration']}")
        print()

    print_split_info("Train", train_guitar, len(train_pairs), train_duration, train_stats)
    print_split_info("Val", val_guitar, len(val_pairs), val_duration, val_stats)

    actual_test_guitars = [g for g in buckets.keys() if g not in [train_guitar, val_guitar]]
    print_split_info("Test", ",".join(actual_test_guitars), len(test_pairs), test_duration, test_stats)

    dataset_split = {
        'train': train_pairs,
        'val': val_pairs,
        'test': test_pairs,
        'stats': {
            'train': train_stats,
            'val': val_stats,
            'test': test_stats
        }
    }

    return dataset_split

try:
    dataset_split = split_by_instrument(pair_files, train_guitar='1', val_guitar='2', test_guitar='3', set_plit=True)
except ValueError as e:
    print(e)

Dataset duration: 44min 59s

Train (guitar 1): 624 pairs  (33m 54s)
  - FS: 156 files, 9m 51s
  - MU: 156 files, 6m 32s
  - ST: 156 files, 8m 37s
  - PK: 156 files, 8m 54s

Val (guitar 2): 134 pairs  (5m 31s)
  - PK: 37 files, 1m 36s
  - ST: 35 files, 1m 17s
  - FS: 30 files, 1m 25s
  - MU: 32 files, 1m 14s

Test (guitar 3): 134 pairs  (5m 34s)
  - MU: 37 files, 1m 22s
  - PK: 31 files, 1m 22s
  - FS: 39 files, 1m 47s
  - ST: 27 files, 1m 3s



## Cell 5 — SegmentDataset + DataLoaders

This code is based on this tutorial: https://pytorch.org/tutorials/beginner/basics/data_tutorial.html

In [17]:
class SegmentDataset(Dataset):
    def __init__(self, pairs, segment_len=SEGMENT_LEN):
        self.segment_len = segment_len
        self.segments_metadata = []

        for clean_path, sat_path in pairs:
            num_frames = sf.info(str(clean_path)).frames

            for start in range(0, num_frames - segment_len + 1, segment_len):
                self.segments_metadata.append({
                    "clean": str(clean_path),
                    "sat":   str(sat_path),
                    "offset": start
                })

    def __len__(self):
        return len(self.segments_metadata)

    def __getitem__(self, idx):
        meta = self.segments_metadata[idx]

        x, _ = torchaudio.load(meta["clean"], frame_offset=meta["offset"], num_frames=self.segment_len)
        y, _ = torchaudio.load(meta["sat"],   frame_offset=meta["offset"], num_frames=self.segment_len)

        x, y = x.squeeze(0), y.squeeze(0)

        return x, y


train_dataset = SegmentDataset(dataset_split['train'])
val_dataset   = SegmentDataset(dataset_split['val'])
test_dataset  = SegmentDataset(dataset_split['test'])

# Create a generator on the correct device for DataLoader shuffling
train_generator = torch.Generator(device=DEVICE)
train_generator.manual_seed(42)

use_pin_memory = torch.cuda.is_available()

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=WORKERS, pin_memory=use_pin_memory, generator=train_generator)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=WORKERS, pin_memory=use_pin_memory)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=WORKERS, pin_memory=use_pin_memory)

print(f"Segments — train: {len(train_dataset)} | val: {len(val_dataset)} | test: {len(test_dataset)}")

Segments — train: 3748 | val: 605 | test: 603


## Cell 6 — LSTM Model

This code is based on these tutorials:

- https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html

- https://pytorch.org/docs/stable/generated/torch.nn.Linear.html

- https://pytorch.org/docs/stable/generated/torch.nn.Module.html


In [18]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, output_size=1, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, skip=SKIP_CON):
        super(LSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.skip = skip

        # Bidirectional=True
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True, bidirectional=True)

        # hidden_size * 2
        self.fc = nn.Linear(hidden_size * 2, output_size)

        self.hidden = None

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lstm_out, self.hidden = self.lstm(x, self.hidden)
        # output = self.h2o(lstm_out)                        
        output = self.fc(lstm_out)
        # skip connection: sums input to output
        if self.skip:
            output = output + x
        return output

    def detach_hidden(self):
        if self.hidden is not None:
            self.hidden = tuple(h.detach() for h in self.hidden)

    def reset_hidden(self):
        self.hidden = None

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


model = LSTMModel().to(device)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

check = torch.zeros(BATCH_SIZE, SEGMENT_LEN, 1)
out = model(check)
print(f"\nInput shape : {check.shape}")
print(f"Output shape: {out.shape}")             
print(f"Hidden h    : {model.hidden[0].shape}")
print(f"Hidden c    : {model.hidden[1].shape}")

LSTMModel(
  (lstm): LSTM(1, 96, num_layers=2, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=192, out_features=1, bias=True)
)

Parameters: 298,945

Input shape : torch.Size([16, 22050, 1])
Output shape: torch.Size([16, 22050, 1])
Hidden h    : torch.Size([4, 16, 96])
Hidden c    : torch.Size([4, 16, 96])


## Cell 7 — Loss functions

To train the models robustly, we use a combined loss function with three factors from the auraloss library [8], addressing emulation fidelity in both the time and frequency domains:

1. **ESR (Error-to-Signal Ratio) with Pre-emphasis**
2. **DC (Direct Current) Loss**
3. **MRSTFT (Multi-Resolution Short-Time Fourier Transform) Loss**

The combined loss is defined as:

$$ \mathcal{L}_{total} = \lambda_{ESR}\mathcal{L}_{ESR\_pe} + \lambda_{DC}\mathcal{L}_{DC} + \lambda_{MRSTFT}\mathcal{L}_{MRSTFT} $$

$\lambda_{ESR}$, $\lambda_{DC}$ e $\lambda_{MRSTFT}$ are the adjustable weights (defined in the **Configuration Cell**) that allow balancing the relative importance of each loss component.

In [19]:
class PreEmphasis(nn.Module):
    def __init__(self, coeffs=(-0.85, 1.0)):
        super().__init__()
        self.filter = nn.Conv1d(1, 1, kernel_size=2, bias=False, padding=0)
        self.filter.weight = nn.Parameter(
            torch.tensor([[list(coeffs)]], dtype=torch.float32),
            requires_grad=False
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)

        x = torch.nn.functional.pad(x, (1, 0))
        x = self.filter(x)

        return x.permute(0, 2, 1)


class CombinedLoss(nn.Module):
    def __init__(self, pre_filt_coeffs=(-0.85, 1.0)):
        super().__init__()
        self.pre_emph   = PreEmphasis(coeffs=pre_filt_coeffs)
        self.esr        = auraloss_time.ESRLoss()
        self.dc         = auraloss_time.DCLoss()
        self.mrstft     = auraloss_freq.MultiResolutionSTFTLoss()

    def forward(self, output, target):
        # Apply PreEmphasis
        out_pe = self.pre_emph(output)
        tgt_pe = self.pre_emph(target)

        output_al = output.permute(0, 2, 1)
        out_pe_al = out_pe.permute(0, 2, 1)
        target_al = target.permute(0, 2, 1)
        tgt_pe_al = tgt_pe.permute(0, 2, 1)

        combined_loss = (LAMBDA_ESR    * self.esr(out_pe_al, tgt_pe_al) +
                         LAMBDA_DC     * self.dc(output_al, target_al)  +
                         LAMBDA_MRSTFT * self.mrstft(output_al, target_al))

        esr_loss = self.esr(out_pe_al, tgt_pe_al).item()
        dc_loss = self.dc(output_al, target_al).item()
        mrstft_loss = self.mrstft(output_al, target_al).item()

        return combined_loss, esr_loss, dc_loss, mrstft_loss


loss_fn = CombinedLoss(pre_filt_coeffs=tuple(PRE_FILT)).to(device)

# Sanity check: output == target -> loss ~0
dummy = torch.randn(BATCH_SIZE, SEGMENT_LEN, 1).to(device)
combined_loss, e_loss, d_loss, m_loss = loss_fn(dummy, dummy)
print(f"Sanity check loss (output==target): {combined_loss:.6f}  (esperado: ~0)")

Sanity check loss (output==target): 0.000000  (esperado: ~0)


## Cell 8 — Optimizer + Scheduler

Adam Optimizer (used in the papers): https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html

Scheduler: ReduceLROnPlateau — https://pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.ReduceLROnPlateau.html


In [20]:
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)
print(f"Optimizer   : Adam  (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"Scheduler   : ReduceLROnPlateau  (factor=0.5, patience=5)")

Optimizer   : Adam  (lr=0.0005, weight_decay=1e-05)
Scheduler   : ReduceLROnPlateau  (factor=0.5, patience=5)


## Cell 9- Training Loop

Source: https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html

In [21]:
if TRAIN:
    log_lines = []
    def log(msg):
        print(msg)
        log_lines.append(msg)

    def format_time(seconds):
        hours = int(seconds // 3600)
        mins = int((seconds % 3600) // 60)
        secs = seconds % 60
        if hours > 0:
            return f"{hours}h {mins}m {secs:.1f}s"
        elif mins > 0:
            return f"{mins}m {secs:.1f}s"
        return f"{secs:.1f}s"

    def train_epoch(model, loader, optimizer, criterion, device, init_len):
        model.train()
        epoch_loss = 0.0
        valid_batches = 0

        for x_batch, y_batch in loader:
            x_batch = x_batch.unsqueeze(-1).to(device)
            y_batch = y_batch.unsqueeze(-1).to(device)

            model.reset_hidden()

            # Warm-up
            if init_len > 0:
                with torch.no_grad():
                    _ = model(x_batch[:, :init_len, :])

            segment = x_batch[:, init_len:, :]
            target  = y_batch[:, init_len:, :]

            # Avoids MRSTFTLoss errors due to too short segments (since it uses a 2048 FFT window)
            if segment.shape[1] < 2048:
                continue

            # Ignores silent segments
            chunk_energy = torch.mean(target ** 2).item()
            if chunk_energy < THRESHOLD:
                continue

            optimizer.zero_grad()
            output = model(segment)
            loss, esr_val, dc_val, mrstft_val = criterion(output, target)

            if torch.isnan(loss) or loss.item() > 20.0:
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()

            epoch_loss += loss.item()
            valid_batches += 1

        return epoch_loss / max(1, valid_batches)


    def validate(model, loader, criterion, device, init_len=INIT_LEN):
        model.eval()
        val_loss = 0.0
        esr_losses = 0.0
        dc_losses = 0.0
        mrstft_losses = 0.0

        with torch.no_grad():
            for x_val, y_val in loader:
                x_val = x_val.unsqueeze(-1).to(device)
                y_val = y_val.unsqueeze(-1).to(device)

                model.reset_hidden()

                if init_len > 0:
                    _ = model(x_val[:, :init_len, :])

                out_val = model(x_val[:, init_len:, :])
                combined_loss, esr_val, dc_val, mrstft_val = criterion(out_val, y_val[:, init_len:, :])

                val_loss += combined_loss.item()
                esr_losses += esr_val
                dc_losses += dc_val
                mrstft_losses += mrstft_val

        avg_val_loss = val_loss / len(loader)
        avg_esr_loss = esr_losses / len(loader)
        avg_dc_loss = dc_losses / len(loader)
        avg_mrstft_loss = mrstft_losses / len(loader)

        return avg_val_loss, avg_esr_loss, avg_dc_loss, avg_mrstft_loss

    # Main training loop com early stopping e scheduler de learning rate
    epoh_target = 200
    best_val_loss   = float('inf')
    patience_count  = 0
    START_EPOCH = 1

    train_losses    = []
    val_losses      = []
    epoh_times      = []
    esr_losses      = []
    dc_losses       = []
    mrstft_losses   = []

    log("Hyperparameters used:")
    log(f" EPOCHS: {EPOCHS} | BATCH_SIZE: {BATCH_SIZE} | VALIDATION_F: {VALIDATION_F} | PATIENCE: {PATIENCE} | LEARNING_RATE: {LEARNING_RATE} | HIDDEN_SIZE: {HIDDEN_SIZE} | NUM_LAYERS: {NUM_LAYERS} | SKIP_CON: {SKIP_CON}\n")

    for epoch in range(START_EPOCH, EPOCHS + 1):
        t0 = time.time()

        # Train
        avg_train_loss = train_epoch(model, train_loader, optimizer, loss_fn, device, INIT_LEN)
        train_losses.append(avg_train_loss)

        # Validation
        if epoch % VALIDATION_F == 0:
            avg_val_loss, avg_esr_loss, avg_dc_loss, avg_mrstft_loss = validate(model, val_loader, loss_fn, device)
            val_losses.append(avg_val_loss)
            esr_losses.append(avg_esr_loss)
            dc_losses.append(avg_dc_loss)
            mrstft_losses.append(avg_mrstft_loss)

            scheduler.step(avg_val_loss)

            log(f"Epoch {epoch:3d}/{EPOCHS} | "
                f"Train: {avg_train_loss:.5f} | "
                f"Time: {format_time(time.time()-t0)} |"
                f"Val: {avg_val_loss:.5f} | "
                f"ESR: {avg_esr_loss:.4f} | "
                f"DC: {avg_dc_loss:.4f} | "
                f"MRSTFT: {avg_mrstft_loss:.4f} | "
                f"LR: {optimizer.param_groups[0]['lr']:.2e} ")

            # Early stopping
            if avg_val_loss < best_val_loss:
                best_val_loss  = avg_val_loss
                patience_count = 0
                torch.save(model.state_dict(), f"{RESULTS_DIR}/best_model.pt")
            else:
                patience_count += 1
                if patience_count >= PATIENCE:
                    log(f"\nEarly stopping na epoch {epoch} (sem melhoria há {PATIENCE} validações).")
                    break
        else:
            log(f"Epoch {epoch:3d}/{EPOCHS} | "
                f"Train: {avg_train_loss:.5f} | "
                f"Time: {format_time(time.time()-t0)}")

        epoh_times.append(time.time() - t0)

    log(f"\nTraining Total Time: {format_time(sum(epoh_times))}")
    log(f"Best validation loss: {best_val_loss:.5f} (epoch {epoch}) — model saved in {RESULTS_DIR}/best_model.pt")

    with open(f"{RESULTS_DIR}/training_log.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(log_lines))
    print(f"Training log saved to {RESULTS_DIR}/training_log.txt")

In [22]:
if TRAIN:
    history = {
        'train_epochs' : len(train_losses),
        'train_losses': train_losses,
        'val_epochs'  : [i * VALIDATION_F for i in range(1, len(val_losses) + 1)],
        'val_losses'  : val_losses,
        'esr_losses'  : esr_losses,
        'dc_losses'   : dc_losses,
        'mrstft_losses': mrstft_losses
    }

    with open(os.path.join(RESULTS_DIR, 'training_history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    print("History saved in Results/training_history.json")

## Cell 10 - Test Set Evaluation

In [23]:
# Loads the best model saved
best_model = LSTMModel().to(device)
best_model.load_state_dict(
    torch.load(f"{RESULTS_DIR}/best_model.pt", map_location=device)
)
best_model.eval()
print(f"Model loaded from {RESULTS_DIR}/best_model.pt")

Model loaded from Results/Results_H96/best_model.pt


In [24]:
if TEST:
        
    pre_emph    = PreEmphasis(coeffs=tuple(PRE_FILT)).to(device)

    def evaluate_test(model, loader, criterion, device):
        model.eval()

        approaches = ["model", "identity", "gain_matched", "static_nonlinear"]
        metrics = {
            name: {"loss_total": 0.0, "loss_esr": 0.0, "loss_dc": 0.0, "loss_mrstft": 0.0}
            for name in approaches
        }
        n_batches  = 0

        with torch.no_grad():
            for x_test, y_test in loader:
                x_test = x_test.unsqueeze(-1).to(device)
                y_test = y_test.unsqueeze(-1).to(device)

                model.reset_hidden()
                output = model(x_test)

                # 1. identity baseline (ouput equals clean input)
                identity_output = x_test

                # 2. gain-matched baseline (RMS energy matching em vez de usar a media, pq no audio a média é ~0)
                rms_target = torch.sqrt(torch.mean(y_test**2) + 1e-8)
                rms_input  = torch.sqrt(torch.mean(x_test**2) + 1e-8)
                gain_matched_output = x_test * (rms_target / rms_input)

                # 3. simple statistical baseline (waveshaping estático com Tanh compensado via RMS)
                static_output = torch.tanh(x_test * 5.0)
                rms_static = torch.sqrt(torch.mean(static_output**2) + 1e-8)
                static_output = static_output * (rms_target / rms_static)

                outputs = {
                    "model": output,
                    "identity": identity_output,
                    "gain_matched": gain_matched_output,
                    "static_nonlinear": static_output
                }

                for name, out in outputs.items():
                    combined_loss, esr_val, dc_val, mrstft_val = criterion(out, y_test)

                    out_al  = out.permute(0, 2, 1)
                    tgt_al  = y_test.permute(0, 2, 1)

                    out_pe  = pre_emph(out).permute(0, 2, 1)
                    tgt_pe  = pre_emph(y_test).permute(0, 2, 1)

                    metrics[name]["loss_total"]   += combined_loss.item()
                    metrics[name]["loss_esr"]     += esr_val
                    metrics[name]["loss_dc"]      += dc_val
                    metrics[name]["loss_mrstft"]  += mrstft_val

                n_batches += 1

        for name in approaches:
            for k in metrics[name]:
                metrics[name][k] /= n_batches

        return metrics

    test_results = evaluate_test(best_model, test_loader, loss_fn, device)

    print("\n── Test Set Evaluation (Model vs Baselines) ──")
    for name, res in test_results.items():
        print(f"[{name.upper()}]")
        print(f"  Loss total   : {res['loss_total']:.5f}")
        print(f"  ESR (pré-ênf): {res['loss_esr']:.5f}")
        print(f"  DC           : {res['loss_dc']:.5f}")
        print(f"  MRSTFT       : {res['loss_mrstft']:.5f}")
        print("────────────────────────────────────────────")


── Test Set Evaluation (Model vs Baselines) ──
[MODEL]
  Loss total   : 0.17612
  ESR (pré-ênf): 0.06156
  DC           : 0.00013
  MRSTFT       : 0.32979
────────────────────────────────────────────
[IDENTITY]
  Loss total   : 1.69835
  ESR (pré-ênf): 1.53323
  DC           : 0.00005
  MRSTFT       : 2.24087
────────────────────────────────────────────
[GAIN_MATCHED]
  Loss total   : 2.12512
  ESR (pré-ênf): 2.28030
  DC           : 0.00010
  MRSTFT       : 2.44215
────────────────────────────────────────────
[STATIC_NONLINEAR]
  Loss total   : 2.11715
  ESR (pré-ênf): 2.27417
  DC           : 0.00229
  MRSTFT       : 2.43011
────────────────────────────────────────────


## Cell 11 - Error analysis by plucking style (FS, MU, PK, ST)

In [25]:
if TEST:

    styles = ["FS", "MU", "PK", "ST"]
    style_metrics = {}

    test_pairs = dataset_split['test']

    print("─── Test Set Evaluation per Plucking Style ───")

    for style in styles:
        style_pairs = [pair for pair in test_pairs if f"_{style}_" in Path(pair[0]).name]

        if len(style_pairs) == 0:
            print(f"\n[{style}] -> No files found in the Test Set.")
            continue

        style_dataset = SegmentDataset(style_pairs)
        style_loader  = DataLoader(style_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=WORKERS, pin_memory=use_pin_memory)

        test_expression_style = evaluate_test(best_model, style_loader, loss_fn, device)

        style_metrics[style] = test_expression_style['model']

        print(f"\n[{style}] - {len(style_pairs)} files found")
        print(f"  Loss total   : {test_expression_style['model']['loss_total']:.5f}")
        print(f"  ESR (pré-ênf): {test_expression_style['model']['loss_esr']:.5f}")
        print(f"  DC           : {test_expression_style['model']['loss_dc']:.5f}")
        print(f"  MRSTFT       : {test_expression_style['model']['loss_mrstft']:.5f}")
        print("────────────────────────────────────────────────")

─── Test Set Evaluation per Plucking Style ───

[FS] - 39 files found
  Loss total   : 0.19810
  ESR (pré-ênf): 0.07623
  DC           : 0.00021
  MRSTFT       : 0.36394
────────────────────────────────────────────────

[MU] - 37 files found
  Loss total   : 0.14928
  ESR (pré-ênf): 0.05551
  DC           : 0.00006
  MRSTFT       : 0.27621
────────────────────────────────────────────────

[PK] - 31 files found
  Loss total   : 0.16961
  ESR (pré-ênf): 0.04884
  DC           : 0.00012
  MRSTFT       : 0.32804
────────────────────────────────────────────────

[ST] - 27 files found
  Loss total   : 0.17799
  ESR (pré-ênf): 0.05784
  DC           : 0.00010
  MRSTFT       : 0.33768
────────────────────────────────────────────────


## Cell 12 - JSON Export (Config + Results)

In [26]:
if TEST:
    
    formatted_results = {}
    for name, res in test_results.items():
        formatted_results[name] = {k: float(f"{v:.5f}") for k, v in res.items()}

    formatted_style_metrics = {}
    for style, metrics in style_metrics.items():
        formatted_style_metrics[style] = {k: float(f"{v:.5f}") for k, v in metrics.items()}

    results_summary = {
        'model': {
            'type'       : 'LSTM',
            'hidden_size': HIDDEN_SIZE,
            'skip_con'   : SKIP_CON,
            'num_layers' : NUM_LAYERS,
            'parameters' : sum(p.numel() for p in best_model.parameters() if p.requires_grad)
        },
        'data': {
            'sample_rate' : SAMPLE_RATE,
            'segment_len' : SEGMENT_LEN,
            'train_size'  : len(train_dataset),
            'val_size'    : len(val_dataset),
            'test_size'   : len(test_dataset)
        },
        'training': {
            'epochs_run' : EPOCHS,
            'batch_size' : BATCH_SIZE,
            'lr_initial' : LEARNING_RATE,
            'loss_weights': {
                'esr': LAMBDA_ESR,
                'dc': LAMBDA_DC,
                'mrstft': LAMBDA_MRSTFT
            },
            'best_epoch'  :  len(val_losses) * VALIDATION_F if TRAIN else None,
        },
        'results': {
            'best_val_loss' : float(f"{best_val_loss:.5f}") if TRAIN else None,
            'test_evaluation' : formatted_results,
            'style_metrics' : formatted_style_metrics
        }
    }

    with open(os.path.join(RESULTS_DIR, 'test_results.json'), 'w') as f:
        json.dump(results_summary, f, indent=2)

    print("Summary saved in Results/test_results.json")

Summary saved in Results/test_results.json


## Cell 13 — Audio Rendering

In [27]:
def render_full_audio(model, clean_path, output_path, warm_up=200):
    model.eval()
    model.reset_hidden()

    clean, sr = torchaudio.load(str(clean_path))
    clean = clean.squeeze(0).to(device)

    with torch.no_grad():
        if warm_up > 0:
            _ = model(clean[:warm_up].reshape(1, -1, 1))

        audio = clean[warm_up:]
        output_audio = model(audio.reshape(1, -1, 1)).squeeze().cpu()

    output_audio = output_audio.unsqueeze(0)
    torchaudio.save(str(output_path), output_audio, sr)
    print(f"Rendered audio saved!")

    return output_audio, sr

# Load a test file
clean_file_preview = Path(CLEAN_DIR + "\ST\\") / "BS_1_EQ_1_ST_NO_1_0.wav"
sat_file_preview = Path(SAT_DIR + "\ST\\") / "BS_1_EQ_1_ST_NO_1_0_saturated.wav"
output_path = os.path.join(RESULTS_DIR, "model_output_melody.wav")

rendered_audio, sr_rendered = render_full_audio(best_model, clean_file_preview, output_path, warm_up=0)

Rendered audio saved!


In [28]:
# preview audio signal clean
clean_waveform, sr_clean = torchaudio.load(clean_file_preview)
print("Original Clean Audio:")
display(Audio(clean_waveform.numpy(), rate=sr_clean))

Original Clean Audio:


In [29]:
# preview audio signal saturado (saturated)
sat_waveform, sr_sat = torchaudio.load(sat_file_preview)
print("Original Saturated Audio:")
display(Audio(sat_waveform.numpy(), rate=sr_sat))

Original Saturated Audio:


In [30]:
# render audio signal preview
print("Predicted Saturated Audio (Model Output):")
display(Audio(rendered_audio.numpy(), rate=sr_rendered))

Predicted Saturated Audio (Model Output):


Renders a single file for each style, using the trained model. The rendered audio is saved in the `Audio_Examples` folder.

The original clean and saturated files are also copied there.

In [31]:
styles = ["FS", "MU", "PK", "ST"]

for style in styles:
    
    AUDIOS_DIR   = Path(CLEAN_DIR).parents[2] / "Audio_Examples" / style
    os.makedirs(AUDIOS_DIR, exist_ok=True)

    clean_file_preview = Path(CLEAN_DIR) / style / f"BS_1_EQ_1_{style}_NO_1_0.wav"
    sat_file_preview = Path(SAT_DIR) / style / f"BS_1_EQ_1_{style}_NO_1_0_saturated.wav"
    output_path = AUDIOS_DIR / f"LSTM_predicted_{style}.wav"
    
    rendered_audio, sr_rendered = render_full_audio(best_model, clean_file_preview, output_path, warm_up=0)
    
    shutil.copy(clean_file_preview, AUDIOS_DIR)
    shutil.copy(sat_file_preview, AUDIOS_DIR)

Rendered audio saved!
Rendered audio saved!
Rendered audio saved!
Rendered audio saved!
